# Project 1 - Explanatory visualization

Information Visualization
---

Resolution by

José Luis Mallqui Roldán, Angel Morales Cirera

Autumn 2025

## Introduction

This Google Colab contains an analysis performed over 3 documents containing data about grant cancellaitons on the United States (US) under Donald Trump's administration.

It is divided into 6 sections, where the first 5 aim to give information related to, but not limited to, some questions proposed by the teachers. The last section is an overall visualization, which aims to provide the answers for all questions, and even a bit more.

## Requirements

These dependencies are needed to run the code. The function bellow should be runned if executing on a local machine (not necessary on Colab).

In [2]:
def install_dependencies():
    %pip install pandas==2.2.2
    %pip install altair==5.5.0
    %pip install vega_datasets==0.9.0
    %pip install streamlit==1.50.0
    %pip install gdown==5.2.0
    %pip install scipy==1.16.3
    %pip install scikit-learn==1.6.1

# Uncomment the following line to install dependencies

#install_dependencies()

# Preprocessing and cleaning

In [2]:
# Download files from Google Drive link

import gdown
import altair as alt


files = {
  'nsf_terminations_airtable.csv': '19PaFWxi2BkgzdLydM6SYEKVvxz-z9jEp',
  'flagged_words_trump_admin.csv': '1ZorwlMbf-_vJcAel33rjTgs42xK8zzEk',
  'cruz_list.csv': '1BZqDomjzcGaKOLkAKnKXcehtNGXhS1HG'
}

for file_name, file_id in files.items():
  gdown.download(f'https://drive.google.com/uc?id={file_id}', file_name, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=19PaFWxi2BkgzdLydM6SYEKVvxz-z9jEp
To: /content/nsf_terminations_airtable.csv
100%|██████████| 7.34M/7.34M [00:00<00:00, 87.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ZorwlMbf-_vJcAel33rjTgs42xK8zzEk
To: /content/flagged_words_trump_admin.csv
100%|██████████| 631/631 [00:00<00:00, 1.83MB/s]
Downloading...
From: https://drive.google.com/uc?id=1BZqDomjzcGaKOLkAKnKXcehtNGXhS1HG
To: /content/cruz_list.csv
100%|██████████| 15.2k/15.2k [00:00<00:00, 5.74MB/s]


Before developing the code to create the visualizations, it is a relevant task to try to ****understand the features of our dataset**** and observe what changes could be made to help us to easily fulfill the comittments of the project. All three files have been examined and proper preprocessing and cleaning methodologies have been applied:

- *flagged_words_trump_admin*: this one is not particularly a file to perform an exhaustive cleaning, however some potential issues have been considered to ensure that we can proceed to the following steps of the project with the conviction of having an adequate content, since the objective of these steps were to create a list of *flagged* terms and create new variables in the termination dataset which allow us to properly respond and prepare further questions.

  **NOTE**: *code developing these steps can be observed in the development of the resolution of Q4*

  For this, we created a new column *text_all* based on the column *project_title* in order to concatenate the title and abstract so we can ****easily compare with simple commands**** whether some words of interest where present in any of these fields, ****lower casing**** was applied to ensure proper matches and ****space normalization**** was also applied, that is, collapsing possible multiple spaces into a single one, with the same purpose of maintaining coherence when looking for words in *text_all*. The following GREL lines chunk was used in OpenRefine for these last tasks, which are equivalent to the lines observed in Q4:


In [3]:
"""
 (cells["project_title"].value.toString() + " " + cells["abstract"].value.toString())
  .toLowercase()
  .replace(/\s+/," ")

"""

<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-2135074348.py:4: SyntaxWarning: invalid escape sequence '\s'
  .replace(/\s+/," ")


'\n (cells["project_title"].value.toString() + " " + cells["abstract"].value.toString())\n  .toLowercase()\n  .replace(/\\s+/," ")\n\n'

  Consequently, to maintain the logic a *text transform* was applied in *text_all* field to clean any apostrophes and normalize additional spaces, resulting in the transformation of 974 cells via these GREL lines, since we found out that some Unicode keys could be giving some trouble.

In [4]:
"""value.toLowercase()
  .replace(/\s+/, " ")
  .replace(/[’]/, "'")
"""

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-2261103875.py:2: SyntaxWarning: invalid escape sequence '\s'
  .replace(/\s+/, " ")


'value.toLowercase()\n  .replace(/\\s+/, " ")\n  .replace(/[’]/, "\'")\n'

On top of that and for Q4 we created a flag_count field containing the number of flagged words in the cancellation grants found in the termination csv since it was thought to be interesting to extract this metric for further analysis. In this case we took advantage of the fact that OpenRefine allowed Python syntax since GREL was giving uncoherent output:

In [5]:
# List based on previous commented modifications
flag_words = [
  "activism","activists","advocacy","advocate","barrier","barriers","bias","biased",
  "bipoc","black and latinx","community diversity","community equity","cultural differences",
  "cultural heritage","culturally responsive","culturally appropriate","disabilities",
  "disability","discrimination","discriminatory","diverse","diversity","diversify",
  "diversifying","equitable","equity","ethnicity","excluded","female","women","gender",
  "hate speech","inclusion","inclusive","lgbtq","lgbt","marginalize","marginalized",
  "minorities","minority","social justice","trauma","underserved","underrepresented",
  "trans","transgender","nonbinary","abortion","climate","emissions","sustainable",
  "decarbonization","carbon footprint","energy transition"
]

"""
txt = value.lower() if value else ""
count = 0
for w in flag_words:
    if w in txt:
        count += 1
return count
"""


'\ntxt = value.lower() if value else ""\ncount = 0\nfor w in flag_words:\n    if w in txt:\n        count += 1\nreturn count\n'

- **cruz_list**: this file contains some grant numbers, and for each one tells if the grant was in Senator Ted's list of grants to be cancelled ("TRUE") or not ("FALSE").

- **nsf_terminations_airtable**: this file is were most of the information was initially contained, and most of it was not going to be used. Some of the columns have been removed, and the information from cruz_list.csv has been added as a new row. Some data insights have also been discovered.

We have found that:
- Columns "terminated" is always "TRUE".
- Columns "suspended" is always "FALSE".
- Column "status" is "Terminated" only if column "reinstated" is "FALSE", and "status" is "Possibly Reinstated" only if column "reinstated" is TRUE.

Therefore, **we have dropped columns "terminated", "suspended" and "status"** because they are redundant.

cruz_list file does not contain all the grants from nsf_terminations_airtable, but because the encodings are TRUE and FALSE, ****we decided not to assume that because a grant is not listed, it means it was not in the list.**** Therefore, we have left the NaNs in the new column in_cruz_list which basically means "unknown". There are also two grants (IDs 2318247 and 2318257) in cruz_list which are not in the nsf_terminations_airtable, both with a "TRUE" value. We have discarded them from the analysis because we have no other information about them in this dataset.

In [6]:
# Preprocessing 1: main dataframe and drop columns

COLUMNS_TO_DROP = ['status', 'terminated', 'suspended', 'termination_date', 'reinstatement_date', 'reinstatement_indicator', 'org_city', 'award_type', 'usa_start_date', 'usa_end_date', 'nsf_start_date', 'nsf_end_date', 'nsf_program_name', 'nsf_primary_program', 'usa_nsf_office', 'nsf_url', 'usaspending_url', 'record_sha1', 'nsf_total_budget', 'nsf_obligated', 'usaspending_obligated', 'usaspending_outlaid', 'estimated_outlays', 'division', 'directorate', 'div', 'dir']

import pandas as pd

nsf_terminations_airtable = pd.read_csv('nsf_terminations_airtable.csv')

nsf_terminations_airtable.drop(columns=COLUMNS_TO_DROP, inplace=True)

In [7]:
# Preprocessing 2: join cruz_list with main dataframe

import pandas as pd

cruz_list = pd.read_csv('cruz_list.csv', sep=";")

data_grants = pd.merge(
    cruz_list,
    nsf_terminations_airtable,
    left_on='grant_number',
    right_on='grant_id',
    how='right' # Keep NaNs only in "in_cruz_list", thus removing 2 missing grants
)
data_grants.drop(columns=['grant_number'],  inplace=True)

# Swap columns to have "grant_id" in the first place
cols = data_grants.columns.tolist()
cols[0], cols[1] = cols[1], cols[0]
data_grants = data_grants[cols]

In [8]:
import pandas as pd

def print_dataframe(df: pd.DataFrame) -> None:
    """Print the full dataframe.
    Used for debugging."""

    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        print(df)

In [9]:
# print_dataframe(data_grants)

In [10]:
# Export the clean dataframe

import pandas as pd

data_grants.to_csv('clean_data.csv', index=False)

----------------------------------------------------------

# Question 1: How are the cancellations distributed by states?

In [11]:
# Q1 preprocessing 1: load codes and names for the states

import pandas as pd

# load the JSON to map the abbrev of US states to FIPS codes
state_codes = pd.read_json(
    'https://gist.githubusercontent.com/wavded/1250983/raw/bf7c1c08f7b1596ca10822baeb8049d7350b0a4b/stateCodeToFips.json',
    orient="values",
    typ="series"
)
state_codes = pd.DataFrame(state_codes,
                   columns=['state_code'])
state_codes['state'] = state_codes.index
state_codes.reset_index(drop=True, inplace=True)

state_names = pd.read_json(
    'https://gist.githubusercontent.com/mshafrir/2646763/raw/8b0dbb93521f5d6889502305335104218454c2bf/states_hash.json',
    orient="values",
    typ="series"
)

state_names = pd.DataFrame(state_names,
                   columns=['state_name'])
state_names['state'] = state_names.index
state_names.reset_index(drop=True, inplace=True)

# merge both DFs to get a single mapping with both FIPS code and state name
state_info = pd.merge(
    state_codes,
    state_names,
    on='state',
    how='inner'
)

#print(state_info)

In [12]:
# Q1 preprocessing 2

import pandas as pd

# cancellations by state
data_grants11 = data_grants.groupby(
    'org_state', as_index=False
).count()[['org_state', 'grant_id']].rename(
    columns={'grant_id':'number_cancellations'}
)

# merge the cancellations per state with state info (FIPS code + name)
data_grants11 = data_grants11.merge(state_info, left_on='org_state', right_on='state', how='outer').drop(columns='org_state')

# fill missing values
data_grants11.fillna(0, inplace=True)

print(data_grants11)

    number_cancellations  state_code state            state_name
0                    4.0           2    AK                Alaska
1                   19.0           1    AL               Alabama
2                    9.0           5    AR              Arkansas
3                    0.0          60    AS        American Samoa
4                   39.0           4    AZ               Arizona
5                  466.0           6    CA            California
6                   52.0           8    CO              Colorado
7                   15.0           9    CT           Connecticut
8                   53.0          11    DC  District Of Columbia
9                    4.0          10    DE              Delaware
10                  59.0          12    FL               Florida
11                  54.0          13    GA               Georgia
12                   0.0          66    GU                  Guam
13                  11.0          15    HI                Hawaii
14                   6.0 

In [13]:
# Q1 Choropleth: just number of cancellations

import altair as alt
from vega_datasets import data as data_vega

map = alt.topo_feature(data_vega.us_10m.url, feature = 'states')

columns = ['number_cancellations', 'state', 'state_name']

chart_cancels = alt.Chart(map).mark_geoshape().properties(
    width = 500,
    height = 300
).project(
    'albersUsa'
).encode(
    alt.Color('number_cancellations:Q',
        scale = alt.Scale(scheme = 'reds'),
        legend = alt.Legend(title = 'Number of cancellations')),
    tooltip=['state:N', 'state_name:N', 'number_cancellations:Q']
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(data_grants11, 'state_code', columns)
)


chart_cancels

alt.Chart(...)

In [14]:
# Q1 Preprocessing 3: download data for analyzing institutions and enrollments

import gdown
import pandas as pd

files = {
  'NCES-IPDES-institutions-24-25.csv': '1O4td_GgEugnCZcTJMUWApuAgyd7b_DcJ',
  'NCES-IPDES-enrollments-23-24.csv': '1Gm-ogSA_IB4CHDnNxTNI-ApmoZCCAVKv'
}

for file_name, file_id in files.items():
  gdown.download(f'https://drive.google.com/uc?id={file_id}', file_name, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1O4td_GgEugnCZcTJMUWApuAgyd7b_DcJ
To: /content/NCES-IPDES-institutions-24-25.csv
100%|██████████| 2.70k/2.70k [00:00<00:00, 7.08MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Gm-ogSA_IB4CHDnNxTNI-ApmoZCCAVKv
To: /content/NCES-IPDES-enrollments-23-24.csv
100%|██████████| 4.83k/4.83k [00:00<00:00, 12.0MB/s]


In [15]:
# Q1 Preprocessing 4: clean and join new data

INSTITUTIONAL_TYPE = "Degree-granting, primarily baccalaureate or above"

import pandas as pd

IPDES_institutions = pd.read_csv('NCES-IPDES-institutions-24-25.csv')
IPDES_enrollments = pd.read_csv('NCES-IPDES-enrollments-23-24.csv')

IPDES_institutions.drop([60, 61], inplace=True)
IPDES_institutions = IPDES_institutions[["State", INSTITUTIONAL_TYPE]] # select relevant column
IPDES_institutions.rename(columns={INSTITUTIONAL_TYPE: 'institutions'}, inplace=True)
IPDES_institutions['institutions'] = IPDES_institutions['institutions'].fillna("1").str.replace(',', '').astype(int)

IPDES_enrollments.drop([60, 61, 62], inplace=True)
IPDES_enrollments = IPDES_enrollments[["State", INSTITUTIONAL_TYPE]] # select relevant column
IPDES_enrollments.rename(columns={INSTITUTIONAL_TYPE: 'enrollments'}, inplace=True)
IPDES_enrollments['enrollments'] = IPDES_enrollments['enrollments'].fillna("1").str.replace(',', '').astype(int)

# combine institutions and enrollments in a single DF by state
edu_data = pd.merge(
    IPDES_institutions,
    IPDES_enrollments,
    on='State',
    how='inner'
)

# combine it with cancellation data by state
data_grants15 = pd.merge(
    data_grants11,
    edu_data,
    left_on='state_name',
    right_on='State',
    how='inner'
).drop(columns=['State']).fillna(1)

# important:
# normalize cancellations by unievrsities/insttitutions
data_grants15['cancellations_per_university'] = data_grants15['number_cancellations'] / data_grants15['institutions']

# normalize by number of students (per every 1000 enrollments)
data_grants15['cancellations_per_1000_enrollments'] = data_grants15['number_cancellations'] / data_grants15['enrollments'] * 1000

In [16]:
# Q1 Choropleth 2: cancellations / universities

import altair as alt
from vega_datasets import data as data_vega

map = alt.topo_feature(data_vega.us_10m.url, feature = 'states')

columns = ['cancellations_per_university', 'state', 'state_name']

chart_cancels = alt.Chart(map).mark_geoshape().properties(
    width = 500,
    height = 300
).project(
    'albersUsa'
).encode(
    alt.Color('cancellations_per_university:Q',
        scale = alt.Scale(scheme = 'reds'),
        legend = alt.Legend(title = 'Cancellations per university')),
    tooltip=['state:N', 'state_name:N', 'cancellations_per_university:Q']
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(data_grants15, 'state_code', columns)
)


chart_cancels

alt.Chart(...)

In [17]:
# Q1 Choropleth 3: cancellations / enrollments

import altair as alt
from vega_datasets import data as data_vega

map = alt.topo_feature(data_vega.us_10m.url, feature = 'states')

columns = ['cancellations_per_1000_enrollments', 'state', 'state_name']

chart_cancels = alt.Chart(map).mark_geoshape().properties(
    width = 500,
    height = 300
).project(
    'albersUsa'
).encode(
    alt.Color('cancellations_per_1000_enrollments:Q',
        scale = alt.Scale(scheme = 'reds'),
        legend = alt.Legend(title = 'Cancellations per 1000 enrollments')),
    tooltip=['state:N', 'state_name:N', 'cancellations_per_1000_enrollments:Q']
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(data_grants15, 'state_code', columns)
)


chart_cancels

alt.Chart(...)

To answer Q1 we first designed a choropleth showing the absolute number of cancelled grants per state, allowing quick identification of outliers: California and Massachusetts. However, raw counts raised more questions due to differing state sizes and number of universities. To improve legibility and the context we incorporated additional data from IPEDS on the number of degree-granting institutions and enrolled students per state, resulting in the creation of two further choropleths: **cancellations per university** and **cancellations per 1,000 enrollments**.

Moreover, we ensured **consistent color coding** with a red sequential palette for intensity, which is distinguishable for color-blind users and enhance the idea of the number of cancellations as something negative. Tooltips provide **exact values and state names**, also enhancing user comprehension. Projection and map height and width were optimized for readability.

This design allows users to answer the question in **multiple ways**: first, by identifying which states have the most cancellations and second, by contextualizing these numbers relative to the number of universities and students, revealing **Massachusetts as the most affected state proportionally**, followed by California. Alternatives such as bar charts per state were discarded due to excessive clutter. Overall these maps reduce are great since they reduce visual noise, highlighting meaningful differences and allowing a great understanding of geographic distribution.

---

# Question 2: What are the institutions that have been more affected in terms of number of cancelled grants? How does this compare to the others?

In [18]:
# Q2 preprocessing 1

import pandas as pd

# group by institution, use count() to obtain number of cancellations
data_grants21 = data_grants.groupby(
    'org_name', as_index=False
).count()[['org_name', 'grant_id']].rename(
    columns={'grant_id':'number_cancellations'}
)

# print_dataframe(data_grants21)

In [19]:
# Q2 Bar chart (discarded)

import altair as alt
import pandas as pd

# too many bars, excessive clutter and zero legibility
alt.Chart(data_grants21).mark_bar().encode(
    alt.X('number_cancellations:Q'),
    alt.Y('org_name:N'),
).properties(title = 'Number of cancelled grants')

alt.Chart(...)

In [20]:
print(data_grants21)

                                              org_name  number_cancellations
0    ALAMO COMMUNITY COLLEGE DISTRICT - St. Philip'...                     1
1                               Alabama A&M University                     4
2                              Albany State University                     2
3                                Allan Hancock College                     1
4                                    Allegheny College                     1
..                                                 ...                   ...
502                     Xavier University of Louisiana                     1
503                    Yakima Valley Community College                     1
504                                    Yale University                     3
505                       Young People's Project, Inc.                     1
506                              ZIKER ENTERPRISES LLC                     1

[507 rows x 2 columns]


In [21]:
# Q2 preprocessing 2

import pandas as pd

# group now by number of cancellations
data_grants22 = data_grants21.groupby(
    'number_cancellations', as_index=False
).count()[['number_cancellations', 'org_name']].rename(
    columns={'org_name':'number_orgs'}
)

In [22]:
# Q2 "histogram" (discarded)

import altair as alt
import pandas as pd

# improved llegibility compared to previous bar chart
# reveals global structure: the majority of inst. have 1-3 cancellations
hist_q2 = alt.Chart(data_grants22).mark_bar().encode(
    x=alt.X(
        'number_cancellations:O',
        #bin=alt.Bin(maxbins=50), does not improve readability
        title='Number of cancelled grants'
    ),
    y=alt.Y(
        'number_orgs:Q',
        title='Number of institutions'
    ),
    tooltip=[
        alt.Tooltip('number_orgs:Q', title='Number of institutions')
    ]
).properties(
    title='Distribution of the number of cancellations',
    width=600,
    height=400
)

hist_q2

alt.Chart(...)

In [23]:
# Q2 box-plot (discarded)

import altair as alt
import pandas as pd

# Solves the problem of having non-uniform increments between bars,
  # and makes it more difficult to miss the outliers
box_q2 = alt.Chart(data_grants22).mark_boxplot(
    outliers=True  # ensures outliers appear as individual dots
).encode(
    x=alt.Y('number_cancellations:Q', title='Number of cancelled grants'),
    tooltip=[
        alt.Tooltip('number_cancellations:Q', title='Cancelled grants')
    ]
).properties(
    title='Distribution of the number of cancellations',
    width=400,
    height=100
)

box_q2


alt.Chart(...)

In [24]:
# Q2 Lollipop chart (discarded)

import altair as alt
import pandas as pd

# filter institutions with >10 cancellations to avoid clutter
data_grants23 = data_grants21.loc[data_grants21['number_cancellations'] > 10]

data_grants23 = data_grants23.sort_values("number_cancellations", ascending=False)


base = alt.Chart(data_grants23).encode(
    y=alt.Y('org_name:N', sort='-x', title='Institution'),
    x=alt.X('number_cancellations:Q', title='Number of cancelled grants')
)


# to visualize it as lollipop (lines)
lines = base.mark_rule(
    color='#999',
    strokeWidth=2
)

# dot of the lollipop
points = base.mark_circle(
    size=120,
    color='#4C78A8'
).encode(
    tooltip=[
        alt.Tooltip('org_name:N', title='Institution'),
        alt.Tooltip('number_cancellations:Q', title='Cancelled grants')
    ]
)

# combine both parts of a lollipop
lollipop_desc = (lines + points).properties(
    title='Q2 — Top institutions by number of cancelled grants (Lollipop chart)',
    width=500,
    height=600
)

lollipop_desc



alt.LayerChart(...)

In [25]:
# Q2 Raincloud

import altair as alt
import pandas as pd


# Color maping
data_grants21['org_category'] = data_grants21['org_name'].apply(
    lambda x: 'UCLA' if x == 'University of California-Los Angeles' else ('HU' if x == 'Harvard University' else 'Other')
)
color_scale = alt.Scale(
    domain=['UCLA', 'HU', 'Other'],
    range=['purple', 'green', 'steelblue']
)
shape_scale = alt.Scale(
    domain=['UCLA', 'HU', 'Other'],
    range=['triangle-up', 'square', 'circle']
)


top_orgs = data_grants21.nlargest(2, 'number_cancellations')

# Density
density = alt.Chart(data_grants21).transform_density(
    'number_cancellations',
    as_=['number_cancellations', 'density']
).mark_area(color='black').encode(
    x=alt.X('number_cancellations:Q', title='Number of cancelled grants'),
    y=alt.Y('density:Q', title='Density')
)

# Points
points = alt.Chart(data_grants21).transform_calculate(
    jitter='-0.25 + (random() - 0.5) * 0.4'
).mark_point(size=30, opacity=0.5).encode(
    x='number_cancellations:Q',
    y=alt.Y('jitter:Q', title='', scale=alt.Scale(domain=[-0.5, 0.4])),
    color=alt.Color('org_category:N',
                    scale=color_scale,
                    legend=alt.Legend(title='Organization')),
    shape=alt.Shape('org_category:N',
                    scale=shape_scale,
                    legend=alt.Legend(title='Organization')),
    tooltip=['org_name', 'number_cancellations']
)

# Combine
raincloud_q2 = (density + points).properties(
    title='Distribution of the number of cancellations per organization',
    width=300,
    height=300
)

# Filter data to x in [0, 40]
raincloud_q2_detail = (
    (density + points)
    .transform_filter(
        (alt.datum.number_cancellations <= 30)
    )
    .properties(
        title='Detail on 0 to 40 cancellations',
        width=230,
        height=300
    )
)

raincloud_overview_detail = (raincloud_q2_detail | raincloud_q2)

raincloud_overview_detail


alt.HConcatChart(...)

To analyse which institutions were **most affected in terms of cancelled grants**, we first aggregated cancellations by organization.

Our initial idea was a bar chart listing every organisation; immediatley discarded due to the high number of bars.

A **histogram** showing how many institutions fall into each cancellation count could reduce the cognitive load, and quickly reveal that: **most institutions suffered only 1-3 cancellations**. Using a single colour palette and tooltips ensures readability and accessibility.

Nevertheless, that histogram has some flaws addressing the non-uniform increment of the values encoded by the bars. We tried to use a box-plot to fix this issue, and allow to catch on the outliers easily.

To study the outliers, we thought of a **lollipop chart** displaying institutions with more than ten cancellations. Lollipops **reduce ink-to-data ratio** and make ranking much more apparent, while allowing **labels and tooltips** without overwhelming the viewer.

At this point, we had an idea to join everything in one visualization: a rainincloud chart with an overview + detail. It provides a complete answer: shows the **overall distribution of cancellations** while **highlighting the specific institutions most impacted**, enabling users to contextualize outliers within the global pattern.

----

# Q3: What are the institutions that have been more affected in terms of budget, and how does this compare to the others?

In [26]:
# Q3 preprocessing 1

import pandas as pd

# aggregate 'estimated_remaining', so it tells how much unspent budget
# each institution had across all its cancelled grants
data_grants31 = data_grants.groupby(
    'org_name', as_index=False
).sum('estimated_remaining')[['org_name', 'estimated_remaining']]

In [27]:
# Q3 preprocessing 2: convert estimated_remaining to millions

import pandas as pd

# with this division we improve readibility
data_grants32 = data_grants31.copy()
data_grants32['estimated_remaining_millions'] = data_grants32['estimated_remaining'] / 1e6

In [28]:
# Q3 histogram

import altair as alt

# preferred over bar charts since binning allows us to observe
# density patterns and better notice outliers
hist_q3_1 = alt.Chart(data_grants32).mark_bar().encode(
    alt.X('estimated_remaining_millions:Q',
          bin=alt.Bin(maxbins=50),
          title='Estimated remaining (million $)'),
    alt.Y('count()', title='Number of organizations')
).properties(title='Distribution of the estimated remaining',
             width=600,
             height=400)

hist_q3_1

alt.Chart(...)

In [29]:
# Q3 preprocessing 3
# obtain only the 15 largets in terms of estimated remaining
import pandas as pd

avg_remaining_millions = data_grants32['estimated_remaining_millions'].mean()
avg_row = pd.DataFrame({
    'org_name': ['Average (all orgs)'],
    'estimated_remaining_millions': [avg_remaining_millions]
})
top_orgs = data_grants32.nlargest(15, 'estimated_remaining_millions')

data_grants33 = pd.concat([top_orgs, avg_row], ignore_index=True)


In [30]:
# Q3 filtered bar chart with average bar
# this way we can observe the two outlier institutions and the reference of
# the average and the other top15

import altair as alt

bars_q3_1 = alt.Chart(data_grants33).mark_bar().encode(
    y=alt.Y('org_name:N', sort='-y', title='Organization'),
    x=alt.X('estimated_remaining_millions:Q', title='Estimated remaining (million $)'),
    color=alt.condition(
        alt.datum.org_name == 'Average (all orgs)',
        alt.value('red'),       # average
        alt.value('steelblue')  # orgs
    ),
    tooltip=['org_name', alt.Tooltip('estimated_remaining_millions:Q', format=".2f")]
).properties(
    title='Top 15 organizations by estimated remaining (in million $)',
    width=600,
    height=400
)

bars_q3_1

alt.Chart(...)

In [31]:
# Q3 preprocessing 4: compute relative_remaining

import pandas as pd

# normalization to compare in relative terms across institutions
data_grants34 = data_grants.copy()
data_grants34['relative_remaining'] = data_grants34['estimated_remaining'] / data_grants34['estimated_budget']

data_grants35 = data_grants34.groupby(
    'org_name', as_index=False
).mean('relative_remaining')[['org_name', 'relative_remaining']]

In [32]:
# Q3 histogram 2
# we observe many institutions with 100% remaining
import altair as alt

hist_q3_2 = alt.Chart(data_grants35).mark_bar().encode(
    alt.X('relative_remaining:Q',
          bin=alt.Bin(maxbins=50),
          title='Relative remaining'),
    alt.Y('count()', title='Number of organizations')
).properties(title='Distribution of the relative remaining',
             width=600,
             height=400)

hist_q3_2


alt.Chart(...)

In [33]:
# Q3 preprocessing 5
# as we previously did, obtain top15 by relative remaining
import pandas as pd

avg_relative_remaining = data_grants35['relative_remaining'].mean()
avg_row = pd.DataFrame({
    'org_name': ['Average (all orgs)'],
    'relative_remaining': [avg_relative_remaining]
})
top_orgs = data_grants35.nlargest(15, 'relative_remaining')

data_grants36 = pd.concat([top_orgs, avg_row], ignore_index=True)


In [34]:
# Q3 filtered bar chart with average bar 2
# visualize some of these institutions with 100% still remaining
# compare it against the average, using hue as a preattentive variable
import altair as alt

bars_q3_2 = alt.Chart(data_grants36).mark_bar().encode(
    y=alt.Y('org_name:N', sort='-y', title='Organization'),
    x=alt.X('relative_remaining:Q', title='Relative remaining'),
    color=alt.condition(
        alt.datum.org_name == 'Average (all orgs)',
        alt.value('red'),       # average
        alt.value('steelblue')  # orgs
    ),
    tooltip=['org_name', alt.Tooltip('relative_remaining:Q', format=".2f")]
).properties(
    title='Top 15 organizations by relative remaining',
    width=600,
    height=400
)

bars_q3_2


alt.Chart(...)

In [35]:
# Q3 preprocessing 6
# focus on Havrard and UCLA, compare now the relative values vs average

import pandas as pd

avg_relative_remaining = data_grants35['relative_remaining'].mean()
avg_row = pd.DataFrame({
    'org_name': ['Average (all orgs)'],
    'relative_remaining': [avg_relative_remaining]
})
data_grants37 = pd.concat([data_grants35, avg_row], ignore_index=True)

data_grants37 = data_grants37[data_grants37['org_name'].isin(['Average (all orgs)', 'Harvard University', 'University of California-Los Angeles'])]

In [36]:
# Q3 filtered bar chart with average bar 3
# we confirm that although both are huge in absolute terms, in relative
# are relatively close to the average, so it might not mean that these
# institutions are necessarily the ones that are more affected
# since there are some smaller ones which due to the cancellation did not
# receive anything.

import altair as alt

bars_q3_3 = alt.Chart(data_grants37).mark_bar().encode(
    y=alt.Y('org_name:N', sort='-y', title='Organization'),
    x=alt.X('relative_remaining:Q', title='Relative remaining'),
    color=alt.condition(
        alt.datum.org_name == 'Average (all orgs)',
        alt.value('red'),       # average
        alt.value('steelblue')  # orgs
    ),
    tooltip=['org_name', alt.Tooltip('relative_remaining:Q', format=".2f")]
).properties(
    title='Harvard, Los Angeles and average - relative remaining budget',
    width=200,
    height=50
)

bars_q3_3


alt.Chart(...)

In [37]:
# Q3 Scatterplot with marginal histograms of the absolute vs relative estimated remaining budget

import altair as alt
import pandas as pd

scatter_data = pd.merge(
    data_grants32[['org_name', 'estimated_remaining_millions']],
    data_grants35[['org_name', 'relative_remaining']],
    on='org_name'
)
shape_scale = alt.Scale(
    domain=['UCLA', 'HU', 'Other'],
    range=['triangle-up', 'square', 'circle']
)

width = 1000
height = 400

x_domain = [-0.25, 1.0]
y_domain = [-5, 100]

# Color maping
scatter_data['org_category'] = scatter_data['org_name'].apply(
    lambda x: 'UCLA' if x == 'University of California-Los Angeles' else ('HU' if x == 'Harvard University' else 'Other')
)
color_scale = alt.Scale(
    domain=['UCLA', 'HU', 'Other'],
    range=['purple', 'green', 'steelblue']
)

# Scatterplot
scatter_plot_q2 = alt.Chart(scatter_data).mark_point(size=60).encode(
    x=alt.X('relative_remaining:Q',
            scale=alt.Scale(domain=x_domain),
            title='Share of the estimated budget across all grants which each organisation has not received'
    ),
    y=alt.Y('estimated_remaining_millions:Q',
            scale=alt.Scale(domain=y_domain),
            title='Millions of $ which the organisations have not received'
    ),
    color=alt.Color('org_category:N',
                    scale=color_scale,
                    legend=alt.Legend(title='Organization')),
    shape=alt.Shape('org_category:N',
                    scale=shape_scale,
                    legend=alt.Legend(title='Organization')),
    tooltip=['org_name', 'estimated_remaining_millions', 'relative_remaining']
).properties(width=width, height=400)

# Top histogram: relative remaining
hist_x = alt.Chart(scatter_data).mark_bar(color='steelblue').encode(
    x=alt.X('relative_remaining:Q', bin=alt.Bin(maxbins=60), scale=alt.Scale(domain=x_domain), title=''),
    y=alt.Y('count()', title=''),
).properties(width=width, height=100)

# Side histogram: absolute remaining
hist_y = alt.Chart(scatter_data).mark_bar(color='steelblue').encode(
    y=alt.Y('estimated_remaining_millions:Q', bin=alt.Bin(maxbins=50), scale=alt.Scale(domain=y_domain), title=''),
    x=alt.X('count()', title=''),
).properties(width=100, height=400)

# Combine scatter + histograms
top = alt.hconcat(hist_x, alt.Chart(pd.DataFrame({'dummy':[0]})).mark_text().encode(text=alt.value('')).properties(width=100))
middle = alt.hconcat(scatter_plot_q2, hist_y)
scatter_marginals_q2 = alt.vconcat(top, middle).resolve_scale(
    x='shared',
    y='shared'
).properties(
    title='Absolute estimated remaining (M $) vs relative remaining'
)

scatter_marginals_q2


alt.VConcatChart(...)

In order to analyse which institutions were most affected in terms of budget, we first aggregated the **estimated remaining amount** per institution. We used a **histogram of the remaining budget** (converted to millions), using 50 bins to reveal the possible underlying distribution. This choice helps users to perceive global patterns: most organizations had little or no remaining budget.

Since absolute amounts depend heavily on institutional size, we computed a relative measure by dividing the remaining budget by the estimated budget. This **normalization** improves comparability across institutions that might have different scales. A second histogram shows a more uniform distribution and reveals a **large cluster of institutions with nearly 100% of funds unspent** when the grants were cancelled.

We also designed **bar charts** highlighting top institutions, including an “average institution” reference bar to support comparative judgement, as the outliers are hard to notice on the histograms. We used a consistent blue–red color scheme and horizontal bars to **reduce label clutter.**

Finally, we decided that a scatterplot with marginal histograms could adress all the goals. It allows users to see the impact in relative and absolute terms and clarify that check outliers are less 'exceptional' once normalized.

----

# Q4: Is there any correlation between the cancelled grants and the list of flagged words?

In [38]:
import altair as alt
import re

# 1. create text_all, concat of title and abstract
data_grants["text_all"] = (
    data_grants["project_title"].fillna("").astype(str) + " " +
    data_grants["abstract"].fillna("").astype(str)
)

# lowercase
data_grants["text_all"] = data_grants["text_all"].str.lower()

# normalize whitespace (collapse multiple spaces to one)
data_grants["text_all"] = data_grants["text_all"].str.replace(r"\s+", " ", regex=True)

# normalize apostrophes
data_grants["text_all"] = data_grants["text_all"].str.replace("[’´`]", "'", regex=True)

# load flagged words
flag_words_df = pd.read_csv("flagged_words_trump_admin.csv")
flag_words = flag_words_df["flagged_word"].str.lower().tolist()


def count_flagged(text):
    """
    Count how manu flagged words appear in full text
    """
    if pd.isna(text):
        return 0
    count = 0
    for w in flag_words:
        # \b handles word boundaries; re.escape ensures safe special chars
        pattern = r"\b{}\b".format(re.escape(w))
        if re.search(pattern, text):
            count += 1
    return count

data_grants["flag_count"] = data_grants["text_all"].apply(count_flagged)

total_grants = len(data_grants)

# store info for each word (count and percentage)
results = []
for w in flag_words:
    count = data_grants["text_all"].str.contains(rf"\b{re.escape(w)}\b", regex=True).sum()
    pct = count / total_grants
    if count > 0:
        results.append({"word": w, "count": count, "pct": pct})

freq_df = pd.DataFrame(results).sort_values("pct", ascending=False)


In [39]:
# dot plot
chart_q4_pct = alt.Chart(freq_df).mark_point(size=90, filled=True).encode(
    x=alt.X("pct:Q", axis=alt.Axis(format="%", title="Relative frequency")),
    y=alt.Y("word:N", sort="-x", title="Flagged word"),
    color=alt.Color("pct:Q", scale=alt.Scale(scheme="reds"), legend=None),
    tooltip=[
        alt.Tooltip("word:N"),
        alt.Tooltip("count:Q", title="Absolute count"),
        alt.Tooltip("pct:Q", title="Relative frequency", format=".2%"),
    ]
).properties(
    title="Relative frequency of each flagged word in terminated grants",
    width=600, height=800
)

chart_q4_pct

alt.Chart(...)

In [40]:
# we observe how many terms contain 'tran' withtin so
# we can properly group all of them which are related
# to the sexual diversity topic and more accuurately
# reflect the cancellations of this group
import re
from collections import Counter

# tokenize text_all in words
all_text = " ".join(data_grants["text_all"].astype(str).tolist())

# extract individual words
words = re.findall(r"[a-zA-Z']+", all_text)

# count how many words have 'trans' in it
trans_related = [w for w in words if "trans" in w]

trans_counts = Counter(trans_related)

trans_df = (
    pd.DataFrame(trans_counts.items(), columns=["word", "count"])
    .sort_values("count", ascending=False)
)

trans_df.head(40)

,word,count
0,transfer,245
1,transformation,216
18,transition,179
2,transform,178
4,transformative,175
22,transport,112
10,transforming,106
20,transitions,104
27,transportation,61
53,translation,45


In [41]:
# plot like previous but now with the grouping of 'trans'
# terms and include an 'all words' dot for comparison


# words to group
trans_whitelist = ["trans", "transgender"]

def contains_flag(text, w):
    if " " in w:
        return w in text
    if w == "trans":
        tokens = re.findall(r"\w*trans\w*", text)
        return any(tok in trans_whitelist for tok in tokens)
    pattern = rf"\b{re.escape(w)}\b"
    return re.search(pattern, text) is not None

def count_flagged(text):
    if pd.isna(text):
        return 0
    return sum(contains_flag(text, w) for w in flag_words)

data_grants["flag_count"] = data_grants["text_all"].apply(count_flagged)

# --- Compute relative document frequency ---
total_grants = len(data_grants)
results = []

for w in flag_words:
    dfreq = data_grants["text_all"].apply(lambda t: contains_flag(t, w)).sum()
    pct = dfreq / total_grants
    if dfreq > 0:
        results.append({"word": w, "count": dfreq, "pct": pct})

freq_df = pd.DataFrame(results).sort_values("pct", ascending=False)

# --- Add "All words" entry ---
all_words_pct = (data_grants["flag_count"] > 0).mean()
freq_df = pd.concat([
    pd.DataFrame([{"word": "All words", "count": (data_grants["flag_count"] > 0).sum(), "pct": all_words_pct}]),
    freq_df
], ignore_index=True)

# --- Visualization with preattentive color for "All words" ---
chart_q4_pct = alt.Chart(freq_df).mark_point(size=90, filled=True).encode(
    x=alt.X("pct:Q", axis=alt.Axis(format="%", title="Relative document frequency")),
    y=alt.Y("word:N", sort="-x", title="Flagged word"),
    color=alt.condition(
        alt.datum.word == "All words",
        alt.value("blue"),      # Highlight "All words" in blue
        alt.Color("pct:Q", scale=alt.Scale(scheme="reds"), legend=None)
    ),
    tooltip=[
        alt.Tooltip("word:N"),
        alt.Tooltip("count:Q", title="# grants where word appears"),
        alt.Tooltip("pct:Q", title="Relative frequency", format=".2%")
    ]
).properties(
    title="Relative frequency of each flagged word in terminated grants",
    width=600, height=800
)

chart_q4_pct

alt.Chart(...)

In [42]:
import altair as alt
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from scipy.cluster.hierarchy import linkage, fcluster


# document-term matrix
vectorizer = CountVectorizer(vocabulary=flag_words, lowercase=True, binary=True)
X = vectorizer.fit_transform(data_grants["text_all"])
df_matrix = pd.DataFrame(X.toarray(), columns=flag_words)
df_matrix = df_matrix.loc[:, (df_matrix.sum(axis=0) > 0)]

# correlation + clustering
cor_matrix = np.corrcoef(df_matrix.T)
cor_matrix = np.nan_to_num(cor_matrix)
Z = linkage(cor_matrix, method='ward')
clusters = fcluster(Z, t=3, criterion='maxclust')
cluster_df = pd.DataFrame({"word": df_matrix.columns, "cluster": clusters})

cluster_names = {
    1: "Equity, Diversity & Sustainability",
    2: "Minorities & Representation",
    3: "Social Justice & Advocacy"
}
cluster_df["cluster_name"] = cluster_df["cluster"].map(cluster_names)

# merge with relative frequency
word_prob_df = cluster_df.merge(freq_df[["word", "pct"]], on="word")
word_prob_df.rename(columns={"pct": "freq_rel"}, inplace=True)


# correction to ensure proper labels  at the vis
visual_order_for_y_axis = [
    "Social Justice & Advocacy",
    "Minorities & Representation",
    "Equity, Diversity & Sustainability"
]

# continuation of the correction
np.random.seed(42)
cluster_index_map_inverted = {
    "Social Justice & Advocacy": 2,
    "Minorities & Representation": 1,
    "Equity, Diversity & Sustainability": 0
}

word_prob_df['cluster_index'] = word_prob_df['cluster_name'].map(cluster_index_map_inverted)
word_prob_df['y_jitter'] = word_prob_df['cluster_index'] + np.random.uniform(-0.2, 0.2, len(word_prob_df))

word_prob_df['x_jitter'] = word_prob_df['freq_rel'] + np.random.uniform(-0.01, 0.01, len(word_prob_df))


y_axis = alt.Y('cluster_name:N', title='Cluster', sort=visual_order_for_y_axis)

# boxplot layer
box = alt.Chart(word_prob_df).mark_boxplot(
    size=40,
    extent='min-max',
    opacity=0.8,
    color='lightblue'
).encode(
    y=y_axis,
    x=alt.X('freq_rel:Q', title='Relative Frequency of Flagged Words')
)

# points layer with vertical jitter and contour
points = alt.Chart(word_prob_df).mark_circle(
    size=80,
    opacity=0.7,
    stroke='black',
    strokeWidth=1
).encode(
    y=alt.Y('y_jitter:Q', axis=None),  # Usando el jitter con el índice invertido
    x='x_jitter:Q',
    tooltip=['word:N', alt.Tooltip('freq_rel:Q', format='.2%')]
)

# combine
raincloud_chart_q4 = alt.layer(box, points).properties(
    title="Relative frequency by flagged word cluster",
    width=700,
    height=400
)

raincloud_chart_q4


alt.LayerChart(...)

To explore whether cancelled grants correlated with the presence of flagged words, we first **normalized and concatenated** the text fields (title + abstract) to ensure consistency in word detection. We applied text **preprocessing: lowercasing, whitespace, and apostrophe normalization to** avoid mismatches. Using regular expressions (re), we computed the document frequency of each flagged term, visualized through a point chart showing their relative occurrence. “All words” was highlighted in blue to ensure perceptual salience while preserving a red color scale consistent with previous questions and **accessible to color-blind users.**

Recognizing the limitation of strict word boundaries, we refined the detection logic for root words like trans, **merging related expressions** such as transgender to improve semantic accuracy. To reveal thematic patterns, we computed co-occurrence correlations among flagged terms and applied Ward’s **hierarchical clustering**, grouping them into three interpretable clusters: Equity, Diversity & Sustainability, Minorities & Representation, and Social Justice & Advocacy.

Finally, we represented the frequency distribution of each cluster using a **raincloud-style boxplot with jittered points**. Horizontal jitter and consistent color encoding enhance legibility and reduced overlap. These design decisions allow users to clearly identify which social topics appeared more frequently in cancelled grants and discover potential bias patterns within the grants.

----

# Q5: Is there any correlation between the cancelled grants and the list of grants in Cruz’s list? And with respect to reinstated grants?

In [43]:
# Q5 preprocessing 1

import pandas as pd

data_grants51 = data_grants.copy()

# aggregate by 'reinstated' and status related to cruz list
data_grants51 = (data_grants51
             .groupby(['reinstated', 'in_cruz_list'])#, dropna=False)
             .size()
             .reset_index(name='number_cancellations'))

data_grants51 = data_grants51.replace({
    'reinstated': {True: 'Reinstated', False: 'Not reinstated'},
    'in_cruz_list': {True: 'In Cruz list', False: 'Out of Cruz list', pd.NA: "Unknown"}
})

# relative percentage of cancellations in each group
data_grants51['percentage_in_group'] = data_grants51.groupby('reinstated')['number_cancellations'].transform(
    lambda x: 100 * x / x.sum()
).round(1).astype(str) + '%'

In [44]:
# Q5 grouped bar chart

import altair as alt

bars = alt.Chart().mark_bar().encode(
    y=alt.Y("in_cruz_list:N",
          title=None),
    x=alt.X('number_cancellations:Q', title='Number of cancellations'),
    color=alt.Color('in_cruz_list:N', title="Is it in Cruz's list?")
).properties(
    width=400,
    height=100
)

# add labels to show perc
texts = bars.mark_text(
    align='left',
    baseline='middle',
    dx=3  # Texts next to the bars
).encode(
    text=alt.Text('percentage_in_group:N')
)

# show two groups (reinstated or not) in different rows
grouped_bars_q5 = alt.layer(
    bars, texts, data=data_grants51
).facet(
    row=alt.Row(
        'reinstated:N', header=alt.Header(
            title=None, labelAngle=0, labelPadding=-30
        ),
    ), title="Cancellations, reinstatemenets and Cruz's list"
)

grouped_bars_q5

alt.FacetChart(...)

In [45]:
# Q5 Bullet chart

import numpy as np
import pandas as pd
import altair as alt

# convert to int to calculate rates afterwards
df = data_grants.copy()
df["reinstated_num"] = df["reinstated"].astype(int)

# split groups
cruz = df[df["in_cruz_list"] == True]["reinstated_num"]
noncruz = df[df["in_cruz_list"] == False]["reinstated_num"]

# means
mean_cruz = cruz.mean()
mean_noncruz = noncruz.mean()

# 95% CI
se = np.sqrt(mean_noncruz * (1 - mean_noncruz) / len(noncruz))
ci_low = max(mean_noncruz - 1.96 * se, 0)
ci_high = min(mean_noncruz + 1.96 * se, 1)

# bullet zones to remark confidence/rejection intervals
zones = pd.DataFrame({
    "zone": ["Lower rejection", "Confidence Interval", "Upper rejection"],
    "y0": [0, ci_low, ci_high],
    "y1": [ci_low, ci_high, 0.15],
    "color": ["#d9d9d9", "#f2f2f2", "#d9d9d9"],
})

# DFs for reference values
value_df = pd.DataFrame({"value": [mean_cruz]})
baseline_df = pd.DataFrame({"baseline": [mean_noncruz]})

# background zones
zones_chart = (
    alt.Chart(zones)
    .mark_bar(width=40)
    .encode(
        y=alt.Y(
            "y0:Q",
            scale=alt.Scale(domain=[0, 0.15]),
            axis=alt.Axis(title="Reinstatement rate", format="%", labelFontSize=12)
        ),
        y2="y1:Q",
        x=alt.value(30),
        color=alt.Color("color:N", scale=None, legend=None)
    )
)

# baseline marker for the mean in Non Cruz
baseline_rule = (
    alt.Chart(baseline_df)
    .mark_rule(color="black", strokeWidth=2)
    .encode(y="baseline:Q")
)

# cruz mean bar
value_bar = (
    alt.Chart(value_df)
    .mark_bar(width=20, color="#377eb8")
    .encode(
        y="value:Q",
        y2=alt.value(0),
        x=alt.value(30),
        tooltip=[alt.Tooltip("value:Q", title="Cruz reinstatement", format=".2%")]
    )
)


# final bullet
bullet_q5 = (
    zones_chart
    + baseline_rule
    + value_bar
).properties(
    title="Vertical Bullet Chart (0–15%)\nReinstatement rate (Cruz List vs 95% CI baseline)",
    width=100,
    height=400
)

bullet = bullet_q5.configure_axis(
    grid=False
)

bullet


alt.LayerChart(...)

To analyze whether Cruz’s list was associated with reinstatement outcomes, we first aggregated cancelled grants by reinstatement and Cruz-list status. The resulting **grouped bar chart** shows absolute counts and relative percentages, helping to assess both class balance and proportional differences. **Bar labels** were included to highlight subtle variations and a consistent blue palette used, then avoiding problematic red–green contrasts and thus **maintaining accessibility**. Faceting by reinstatement status **improved readability** and **reduced clutter** while preserving visual comparability.

Since the first chart suggested no strong association, we explored differences in reinstatement probability using a **vertical bullet chart**. This representation encodes in an **efficient** manner statistical context: background zones mark **95% confidence intervals** for the non-Cruz group, while a blue bar represents the Cruz-list mean. A black reference line enhances **perceptual hierarchy**, allowing viewers to rapidly compare whether differences fall within the expected range.

All visual encodings follow a **consistent color scheme** and minimalistic design to improve clarity and interpretability. The results reveal that grants on Cruz’s list had similar reinstatement rates to others, with no statistically significant deviation—findings that are perceptually clear even for non-expert audiences thanks to normalization, explicit labels, and effective visual hierarchy.

# VFinal visualization

In [46]:
# FINAL Choropleth x3

import altair as alt
from vega_datasets import data as data_vega


map = alt.topo_feature(data_vega.us_10m.url, feature = 'states')

tooltip_vars = ['number_cancellations', 'cancellations_per_university', 'cancellations_per_1000_enrollments', 'state', 'state_name']
variable_list = ['number_cancellations', 'cancellations_per_university', 'cancellations_per_1000_enrollments']

chart_cancels_x3 = alt.Chart(map).mark_geoshape().properties(
    width = 250,
    height = 150
).project(
    'albersUsa'
).encode(
    alt.Color(alt.repeat('row'),
        type='quantitative',
        scale = alt.Scale(scheme = 'reds'),
        #legend=alt.Legend(title='')
    ),
    tooltip=['state:N', 'state_name:N', 'number_cancellations:Q', 'cancellations_per_university:Q', 'cancellations_per_1000_enrollments:Q']
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(data_grants15, 'state_code', tooltip_vars)
).repeat(
    row = variable_list
).resolve_scale(
    color='independent'
)


chart_cancels_x3

alt.RepeatChart(...)

In [47]:
raincloud_q2 = raincloud_q2.properties(
    width = 600,
    height = 200
)
chart_q4_pct = chart_q4_pct.properties(
    width = 600,
    height = 700
)
raincloud_chart_q4 = raincloud_chart_q4.properties(
    width = 600,
    height = 223
)
bullet_q5 = bullet_q5.properties(
    title = "Cruz vs Non-Cruz list grants mean rate comparison with CI regions (a = 0.05)",
    width = 50,
    height = 223
)


final_vis = (((raincloud_overview_detail & chart_q4_pct).resolve_scale(color='independent', shape='independent') | ((scatter_marginals_q2 | chart_cancels_x3).resolve_scale(color='independent', shape='independent') & (raincloud_chart_q4 | grouped_bars_q5 | bullet_q5).resolve_scale(color='independent', shape='independent')))).resolve_scale(
    color='independent', shape='independent'
)
final_vis.properties(title = "Who has been more affected by the cancellation of grants under Trump's adminstration?")

alt.HConcatChart(...)

The final visualization integrates multiple standpoints to provide a full picture of cancellation patterns and their possible causes.
A **histogram with overlaid scatter points** displays the distribution of cancelled grants, then combining density and raw counts to assess both macro and micro level interpretation. A **dot plot** summarizes the relative frequency of flagged terms plus an aggregate “all words” point revealing semantic biases among terminated projects.

**Two additional histograms** compare absolute and relative remaining budgets, clarifying how institutional size influences perceived impact. A **scatter plot** of unreceived budget share versus total amount uncovers whether large losses are concentrated in a few organizations.

To examine topic sensitivity a **raincloud plot** visualizes the distribution of flagged-word frequencies across semantic clusters, blending density and jittered points to avoid overlap and emphasize variability.

Two **grouped horizontal bar charts** summarize reinstatement and Cruz-list interactions, maintaining a consistent palette considering accessibility. Three U.S. choropleths finally contextualize these findings geographically, normalizing by total institutions and enrollments to observe data from different perspectives.